# phase2-convnext-s-4fold-v2 — R7-beater
ConvNeXt-S + per-finding attention MIL + CORAL ordinal heads + fastMRI aux + Gemini severity + 4-fold CV. Set FOLD=0..3 and PROBE=False for a full run.

In [ ]:
# ================= FAIL-FAST + CONFIG (literal top) =================
# VERSION prints first. Then FOLD, backbone+weights, label/severity discovery.
FOLD = 0
VERSION = "phase2-convnext-s-4fold-v3 (2026-10-14: v2 resume + CORAL ordinal inference boost on 4 severity findings)"
print(f"VERSION: {VERSION}")
print(f"FOLD: {FOLD}")
assert FOLD in (0, 1, 2, 3), f"FATAL: FOLD={FOLD} must be in [0,1,2,3]"

import timm as _timm_ff
assert 'convnext_small' in _timm_ff.list_models(), \
    'FATAL: convnext_small not in timm %s (check Internet / timm version)' % _timm_ff.__version__
print(f"fail-fast: convnext_small found in timm {_timm_ff.__version__}")
print('fail-fast: downloading convnext_small pretrained weights (verify-first)...')
_ff_m = _timm_ff.create_model('convnext_small', pretrained=True, num_classes=0,
                              drop_path_rate=0.1)
_ff_n = sum(p.numel() for p in _ff_m.parameters()) / 1e6
assert _ff_m.num_features == 768, f"FATAL: expected 768-dim features, got {_ff_m.num_features}"
print(f"fail-fast: pretrained weights OK ({_ff_n:.1f}M params, 768-dim features)")
del _ff_m, _ff_n, _timm_ff

import os
LABEL_CSV = "/kaggle/input/datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv"
if not os.path.isfile(LABEL_CSV):
    raise RuntimeError(f"FATAL: {LABEL_CSV} not found. Attach stevenleehans/rsna-knee-llm-report-labels dataset.")
print(f"fail-fast: teacher labels OK: {LABEL_CSV}")

_sev_path = "/kaggle/input/datasets/sridharshivashankar/severity-all-4276-json/severity_all_4276.json"
if os.path.isfile(_sev_path):
    SEV_SOURCE = ("combined", _sev_path)
    print(f"fail-fast: severity grades OK (combined): {SEV_SOURCE[1]}")
else:
    raise RuntimeError(f"FATAL: {_sev_path} not found. Attach sridharshivashankar/severity-all-4276-json dataset.")
print("fail-fast: ALL GREEN")

# ================= CONFIG (every knob lives here) =================
PROBE = True              # True: tiny subset, validate pipeline. False: full fold run.
PROBE_N_STUDIES = 50
PROBE_EPOCHS = 3

# --- data ---
IMAGE_SIZE = 288          # ConvNeXt-S @ 288px (cache is 336, resize in dataset)
N_SLICES = 8              # uniform slices sampled per plane
PLANES = ['axial', 'coronal', 'sagittal']

# --- model ---
BACKBONE_NAME = 'convnext_small'  # timm, in12k_ft_in1k weights (crash loud if missing)
PRETRAINED = True
DROP_PATH_RATE = 0.1      # R7 recipe
ATTN_DIM = 256            # nartaa spec: per-finding attention 768->256->12
N_FINDINGS = 12
N_CORAL_THRESHOLDS = 3    # grades 0-3 -> P(y>0), P(y>1), P(y>2)

# --- training ---
BATCH_SIZE = 2            # 2 studies x 24 images = 48/batch @ 288px (probe verifies fit)
ACCUM_STEPS = 8           # effective batch = 16
EPOCHS = 30
PATIENCE = 8              # early stopping on val macro AUC
LR_BACKBONE = 1e-4        # backbone base LR (per-stage decay below)
LR_STAGE_DECAY = 0.9      # stage N gets LR * 0.9^(3-N)
LR_HEAD = 1e-3            # MIL + classifiers
WEIGHT_DECAY = 0.02       # not on biases/norms/1-D
WARMUP_FRAC = 0.10        # linear warmup, then cosine decay to 0
GRAD_CLIP = 1.0
EMA_DECAY = 0.998
SWA_LAST = 3              # average last 3 EMA snapshots
USE_AMP = True
FREEZE_BN = True
GOLD_UPWEIGHT = 8.0

# --- ASL loss ---
ASL_GAMMA_POS = 0.0
ASL_GAMMA_NEG = 4.0
ASL_MARGIN = 0.05

# --- auxiliary losses (the upgrades) ---
LAMBDA_CORAL = 0.3        # CORAL ordinal aux weight (Justin's edge: real 0-3 grades)
FASTMRI_AUX_W = 0.2       # fastMRI lesion aux weight (R7 trick)
FMRI_POS_WEIGHT = 8.0     # R7: BCE pos_weight 8 for lesion detection

# --- folds ---
N_FOLDS = 4
VAL_FOLD = FOLD           # this run validates on fold FOLD
SEED = 123                # differs from Tiny (42) and CoAtNet (0)

# --- io ---
CKPT_DIR = '/kaggle/working/convnext_s_4fold'

LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
# CORAL severity findings, aligned to SEV_KEYS order below
CORAL_FINDING_IDX = [LABELS.index('Synovitis'), LABELS.index('PF OA'),
                     LABELS.index('Medial OA'), LABELS.index('Lateral OA')]
SEV_KEYS = ['synovitis', 'pfoa', 'medial_oa', 'lateral_oa']

print('CONFIG DUMP')
for k in ['FOLD','VAL_FOLD','PROBE','IMAGE_SIZE','BACKBONE_NAME','DROP_PATH_RATE','ATTN_DIM',
          'BATCH_SIZE','ACCUM_STEPS','EPOCHS','PATIENCE','LR_BACKBONE','LR_HEAD',
          'LAMBDA_CORAL','FASTMRI_AUX_W','N_FOLDS','SEED','CKPT_DIR']:
    print(f'  {k} = {globals()[k]}')
print(f'  CORAL_FINDING_IDX = {CORAL_FINDING_IDX} (synovitis, pfoa, medial_oa, lateral_oa)')



In [ ]:
# ================= SETUP: imports, device, seeds, attachments =================
import os, sys, json, time, math, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print(f'torch {torch.__version__} | cuda: {torch.cuda.is_available()}',
      f' | gpus: {torch.cuda.device_count()}')
assert torch.cuda.is_available(), 'FATAL: no GPU'
NGPU = torch.cuda.device_count()
for i in range(NGPU):
    print(f'  gpu{i}: {torch.cuda.get_device_name(i)}')
DEVICE = torch.device('cuda:0')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# ---- attachment check (hardcoded paths; no recursive scan) ----
print('\nattachment check (hardcoded paths):')
import glob as _ag
for _d in ['phase0runa', 'phase0runb', 'phase0runc']:
    _hits = _ag.glob(f'/kaggle/input/datasets/sridharshivashankar/{_d}/*/manifest.json')
    print(f'  manifest.json ({_d}): {len(_hits)} hit(s)')
    for _h in _hits[:1]:
        print(f'    {_h}')
print('  llm_labels_v4_blend.csv: 1 hit(s)')
print(f'    {LABEL_CSV}')
print('  train.csv: 1 hit(s)')
print('    /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
print(f'  severity: {SEV_SOURCE[0]} -> {SEV_SOURCE[1]}')

import glob as _mg
MANIFEST_PATHS = []
for _d in ['phase0runa', 'phase0runb', 'phase0runc']:
    MANIFEST_PATHS += _mg.glob(f'/kaggle/input/datasets/sridharshivashankar/{_d}/*/manifest.json')
assert MANIFEST_PATHS, 'FATAL: manifest.json not found — attach phase0 datasets'

TRAIN_CSV = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv'
assert os.path.isfile(TRAIN_CSV), f'FATAL: {TRAIN_CSV} not found' 
print(f'manifests: {len(MANIFEST_PATHS)} | label csv: {LABEL_CSV}')
print(f'train.csv: {TRAIN_CSV} (for gold labels + priors)')
os.makedirs(CKPT_DIR, exist_ok=True)
assert os.path.exists(LABEL_CSV), f'FATAL: label CSV missing: {LABEL_CSV}'

# ---- 9h-limit resume: restore _last.pt from hardcoded path ----
_rs_prefix = f'convnext_s_fold{FOLD}'
_rs_dst = os.path.join(CKPT_DIR, f'{_rs_prefix}_last.pt')
os.makedirs(CKPT_DIR, exist_ok=True)
_rs_found = None
# HARDCODED checkpoint path (no recursive search)
_hardcoded_ckpt = '/kaggle/input/datasets/sridharshivashankar/fold0-checkpoint/convnext_s_4fold/convnext_s_fold0_last.pt'
if os.path.exists(_hardcoded_ckpt):
    _rs_found = _hardcoded_ckpt
    print(f'resume: found hardcoded {_rs_found}')
else:
    print('resume: hardcoded checkpoint not found (fresh run)')
if _rs_found:
    import shutil as _rs
    _rs.copy(_rs_found, _rs_dst)
    print(f'resume: copied {_rs_found} -> {_rs_dst}')
else:
    print('resume: no checkpoint to restore')
print('setup: ALL GREEN')






In [ ]:
# ================= CORE MATH (tested; do not modify without re-testing) =================
def asl_loss(logits, targets, gamma_pos=ASL_GAMMA_POS, gamma_neg=ASL_GAMMA_NEG,
             margin=ASL_MARGIN, eps=1e-8):
    # logits, targets: (B, 12). Soft targets supported. Returns (B, 12) element losses.
    p = torch.sigmoid(logits).clamp(eps, 1 - eps)
    loss_pos = -((1 - p) ** gamma_pos) * torch.log(p)
    p_m = (p - margin).clamp(min=0).clamp(eps, 1 - eps)
    loss_neg = -((p_m) ** gamma_neg) * torch.log(1 - p_m)
    return targets * loss_pos + (1 - targets) * loss_neg


def auc_score(y_true, y_score):
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    pos = y_score[y_true == 1]; neg = y_score[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    n1, n0 = len(pos), len(neg)
    labs = np.concatenate([np.ones(n1), np.zeros(n0)])
    sc = np.concatenate([pos, neg])
    order = np.argsort(sc, kind='mergesort')
    s_sorted = sc[order]
    ranks = np.empty(len(sc))
    i = 0
    while i < len(sc):
        j = i
        while j < len(sc) and s_sorted[j] == s_sorted[i]:
            j += 1
        ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    r1 = ranks[labs == 1].sum()
    return (r1 - n1 * (n1 + 1) / 2.0) / (n1 * n0)


def macro_auc(y_true_2d, y_score_2d, labels=LABELS):
    aucs = {}
    for j, lab in enumerate(labels):
        a = auc_score(y_true_2d[:, j], y_score_2d[:, j])
        aucs[lab] = a
    valid = [a for a in aucs.values() if a is not None]
    return (sum(valid) / len(valid) if valid else None), aucs


def grade_to_coral_targets(grades):
    # grades: (...,) float tensor with values 0..3 (NaN -> all-False, masked elsewhere).
    # Returns (..., 3) float tensor: [y>0, y>1, y>2].
    g = grades.unsqueeze(-1)  # (..., 1)
    thresh = torch.tensor([0.5, 1.5, 2.5], dtype=grades.dtype, device=grades.device)
    return (g > thresh).float()


def coral_loss(coral_logits, sev_targets, sev_mask):
    # coral_logits: (B, 4, 3), sev_targets: (B, 4, 3) float, sev_mask: (B, 4) float.
    # BCE on each threshold, masked per finding. Returns scalar.
    bce = F.binary_cross_entropy_with_logits(coral_logits, sev_targets, reduction='none')
    m = sev_mask.unsqueeze(-1)  # (B,4,1) broadcast over thresholds
    per_finding = (bce * m).sum(-1) / m.sum(-1).clamp(min=1e-6)  # (B,4)
    n_valid = (sev_mask.sum(-1) > 0).float()  # (B,) studies with >=1 grade
    per_study = per_finding.sum(-1) / sev_mask.sum(-1).clamp(min=1e-6)  # (B,)
    return (per_study * n_valid).sum() / n_valid.sum().clamp(min=1e-6)


def scanner_group_key(study_uid, n_parts=8):
    return '.'.join(study_uid.split('.')[:n_parts])


def assign_folds(study_uids, n_folds=N_FOLDS, seed=SEED):
    rng = np.random.RandomState(seed)
    groups = {}
    for uid in study_uids:
        groups.setdefault(scanner_group_key(uid), []).append(uid)
    keys = sorted(groups.keys())
    rng.shuffle(keys)
    fold_of = {}
    for i, k in enumerate(keys):
        for uid in groups[k]:
            fold_of[uid] = i % n_folds
    return fold_of, groups


# self-tests
_lt = torch.tensor([[0.0, 2.0, -2.0]]); _tg = torch.tensor([[1.0, 1.0, 0.0]])
_L = asl_loss(_lt, _tg)
assert _L[0, 1] < _L[0, 0] and _L[0, 2] < _L[0, 0], 'ASL focusing broken'
assert abs(auc_score([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]) - 1.0) < 1e-9
assert abs(auc_score([0, 1], [0.5, 0.5]) - 0.5) < 1e-9
assert auc_score([1, 1], [0.5, 0.6]) is None
_gt = torch.tensor([0.0, 1.0, 2.0, 3.0])
_ct = grade_to_coral_targets(_gt)
_exp = torch.tensor([[0., 0., 0.], [1., 0., 0.], [1., 1., 0.], [1., 1., 1.]])
assert torch.equal(_ct, _exp), f'CORAL target conversion broken: {_ct}'
_pl = torch.tensor([[[10., 10., 10.], [-10., -10., -10.],
                     [10., 10., -10.], [10., -10., -10.]]])
_pt = torch.tensor([[[1., 1., 1.], [0., 0., 0.],
                     [1., 1., 0.], [1., 0., 0.]]])
_pm = torch.ones(1, 4)
assert coral_loss(_pl, _pt, _pm).item() < 1e-3, 'CORAL loss broken'
assert coral_loss(_pl, _pt, torch.zeros(1, 4)).item() == 0.0, 'masked CORAL should be 0'
print('core math self-test: ALL GREEN (ASL + AUC + CORAL)')


In [ ]:
# ================= DATA LOADING: cache index =================
class ShardStore:
    def __init__(self, manifest_paths):
        self.studies = {}
        self._z = {}
        for mp in manifest_paths:
            cache_dir = str(Path(mp).parent)
            man = json.load(open(mp))
            for uid, entry in man['studies'].items():
                if uid in self.studies:
                    raise RuntimeError(f'duplicate study {uid} across caches')
                self.studies[uid] = (cache_dir, entry)
        print(f'indexed {len(self.studies)} studies from {len(manifest_paths)} manifests')

    def get_plane(self, uid, plane):
        pid = os.getpid()
        cache_dir, entry = self.studies[uid]
        shard = entry['shard']
        key = (pid, cache_dir, shard)
        z = self._z.get(key)
        if z is None:
            for k in [k for k in self._z if k[0] != pid]:
                try:
                    self._z.pop(k).close()
                except Exception:
                    pass
            z = np.load(os.path.join(cache_dir, shard))
            self._z[key] = z
            while len(self._z) > 4:
                k = next(iter(self._z))
                try:
                    self._z.pop(k).close()
                except Exception:
                    pass
        return z[f'{uid}/{plane}']

    def planes_of(self, uid):
        return list(self.studies[uid][1]['planes'].keys())


store = ShardStore(MANIFEST_PATHS)
from collections import Counter
pc = Counter()
for uid in store.studies:
    pc[tuple(sorted(store.planes_of(uid)))] += 1
print('plane-set distribution:')
for k, v in pc.most_common():
    print(f'  {k}: {v}')
n_missing = sum(1 for uid in store.studies if len(store.planes_of(uid)) < 3)
print(f'studies with <3 planes: {n_missing} (tolerated, never dropped)')


In [ ]:
# ================= LABELS: teacher (8) + Gemini severity (4) + gold override =================
# SOFT LABELS (upgrade #3): the teacher's continuous values are used DIRECTLY —
# never thresholded. nartaa's recipe ("0.8 for suspected") is preserved via the
# confidence weight w = 2|p-0.5|: uncertain cells contribute less, not zero signal.
lab_df = pd.read_csv(LABEL_CSV, encoding='latin-1')
print(f'label csv: {lab_df.shape}')

def _norm(c):
    return str(c).strip().casefold()
_norm_map = {}
for c in lab_df.columns:
    n = _norm(c)
    if n in _norm_map:
        raise RuntimeError(f'FATAL: duplicate label column after normalization: {c!r}')
    _norm_map[n] = c
_expected = ['StudyInstanceUID'] + LABELS
_missing = [e for e in _expected if _norm(e) not in _norm_map]
if _missing:
    raise RuntimeError(f'FATAL: label CSV missing columns {sorted(set(_missing))}; '
                       f'actual header: {list(lab_df.columns)}')
lab_df = lab_df[[_norm_map[_norm(e)] for e in _expected]]
lab_df.columns = _expected
print(f'  header normalized: StudyInstanceUID + {len(LABELS)} findings')
assert lab_df[LABELS].notna().all().all(), 'FATAL: NaN in label csv'
assert ((lab_df[LABELS] >= 0) & (lab_df[LABELS] <= 1)).all().all(), 'FATAL: labels outside [0,1]'

# ---- severity grades -> ordinals ----
GRADE_TO_ORD = {'none': 0, 'mild': 1, 'moderate': 2, 'severe': 3}
def _grade_to_ord(v):
    if v is None:
        return np.nan
    if isinstance(v, bool):
        return np.nan
    if isinstance(v, (int, np.integer)):
        return float(v) if 0 <= int(v) <= 3 else np.nan
    if isinstance(v, float):
        return v if (not np.isnan(v) and v.is_integer() and 0 <= v <= 3) else np.nan
    s = str(v).strip().lower()
    return float(GRADE_TO_ORD[s]) if s in GRADE_TO_ORD else np.nan

sev_grades = {}  # StudyInstanceUID -> (4,) float32 ordinals, NaN = missing
def _parse_sev_row(r):
    uid = r.get('StudyInstanceUID')
    if not uid:
        return
    sev_grades[str(uid)] = np.array([_grade_to_ord(r.get(k)) for k in SEV_KEYS],
                                    dtype=np.float32)

if SEV_SOURCE[0] == 'combined':
    with open(SEV_SOURCE[1]) as f:
        _data = json.load(f)
    _rows = _data.values() if isinstance(_data, dict) else _data
    for _r in _rows:
        _parse_sev_row(_r)
else:
    for _fp in SEV_SOURCE[1]:
        with open(_fp) as f:
            _data = json.load(f)
        _rows = _data if isinstance(_data, list) else list(_data.values())
        for _r in _rows:
            _parse_sev_row(_r)
print(f'severity grades loaded: {len(sev_grades)} studies')
from collections import Counter as _C
for j, k in enumerate(SEV_KEYS):
    _c = _C(int(v[j]) for v in sev_grades.values() if not np.isnan(v[j]))
    print(f'  {k}: ' + ', '.join(f'{["none","mild","mod","sev"][g]}={_c[g]}' for g in sorted(_c)))

# ---- gold ----
gold = {}
priors = None
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
    tr = pd.read_csv(TRAIN_CSV, encoding='latin-1', usecols=['StudyInstanceUID'] + LABELS)
    gm = tr[LABELS].notna().all(axis=1)
    gdf = tr[gm]
    print(f'gold studies in train.csv: {len(gdf)}')
    for _, r in gdf.iterrows():
        gold[str(r['StudyInstanceUID'])] = r[LABELS].values.astype(np.float32)
    priors = gdf[LABELS].mean().values.astype(np.float32)
else:
    print('WARNING: train.csv not found — gold override disabled')
    priors = lab_df[LABELS].mean().values.astype(np.float32)
print('priors:', dict(zip(LABELS, np.round(priors, 3))))

# ---- build packs: (targets[12], loss_mask[12], sample_weight, is_gold, ord[4], ord_mask[4]) ----
lab_idx = lab_df.set_index('StudyInstanceUID')
_missing_teacher = [u for u in store.studies
                    if str(u) not in lab_idx.index and str(u) not in gold]
assert not _missing_teacher, f'FATAL: {len(_missing_teacher)} studies missing from teacher CSV'
label_pack = {}
n_gold_hit = n_sev_bin = 0
for uid in store.studies:
    suid = str(uid)
    if suid in gold:
        targets = gold[suid].copy()
        mask = np.ones(12, np.float32)
        w, is_gold = GOLD_UPWEIGHT, True
    else:
        targets = lab_idx.loc[suid, LABELS].values.astype(np.float32)
        mask = (2.0 * np.abs(targets - 0.5)).astype(np.float32)  # soft-label confidence
        w, is_gold = 1.0, False
    ord_g = np.full(4, np.nan, dtype=np.float32)
    ord_m = np.zeros(4, dtype=np.float32)
    if suid in sev_grades:
        g = sev_grades[suid]
        valid = ~np.isnan(g)
        if valid.any():
            if not is_gold:
                # Gemini binary overwrite (100% on gold for binary): grade>=1 -> 1
                for j, li in enumerate(CORAL_FINDING_IDX):
                    if valid[j]:
                        targets[li] = 1.0 if g[j] >= 1 else 0.0
                        mask[li] = 1.0
                        n_sev_bin += 1
            ord_g = np.where(valid, g, np.nan).astype(np.float32)
            ord_m = valid.astype(np.float32)
    label_pack[uid] = (targets.astype(np.float32), mask, float(w), bool(is_gold),
                       ord_g, ord_m)
    if is_gold:
        n_gold_hit += 1
print(f'label packs: {len(label_pack)} | gold override: {n_gold_hit} | '
      f'Gemini binary severity cells: {n_sev_bin}')
n_ord = sum(1 for v in label_pack.values() if v[5].sum() > 0)
print(f'studies with ordinal grades for CORAL aux: {n_ord}')


In [ ]:
# ================= FOLDS: 4-fold, seeded (UID-prefix proxy collapses) =================
all_uids = sorted(store.studies.keys())
fold_of, groups = assign_folds(all_uids)
import random as _r
if len(groups) < 2:
    print(f'WARNING: only {len(groups)} scanner-proxy group(s); FALLBACK to seeded random folds')
    _rng = _r.Random(SEED)
    _shuf = sorted(all_uids)
    _rng.shuffle(_shuf)
    fold_of = {uid: (idx % N_FOLDS) for idx, uid in enumerate(_shuf)}
print(f'{len(groups)} groups -> {N_FOLDS} folds')
for f in range(N_FOLDS):
    members = [u for u in all_uids if fold_of[u] == f]
    g_gold = sum(1 for u in members if label_pack[u][3])
    g_sev = sum(1 for u in members if label_pack[u][5].sum() > 0)
    print(f'  fold {f}: {len(members)} studies, {g_gold} gold, {g_sev} w/ severity grades')
train_uids = [u for u in all_uids if fold_of[u] != VAL_FOLD]
val_uids = [u for u in all_uids if fold_of[u] == VAL_FOLD]
print(f'FOLD={FOLD} (val fold {VAL_FOLD}) | train: {len(train_uids)} | val: {len(val_uids)}')
assert len(val_uids) > 0 and len(train_uids) > 0, 'FATAL: empty train/val split'
if PROBE:
    train_uids = train_uids[:PROBE_N_STUDIES]
    val_uids = val_uids[:max(4, PROBE_N_STUDIES // 4)]
    print(f'PROBE subset -> train: {len(train_uids)} | val: {len(val_uids)}')


In [ ]:
# ================= DATASET: heavy augmentation (no flips) =================
def sample_slices(vol, n):
    S = vol.shape[0]
    if S == n:
        return vol
    idx = np.linspace(0, S - 1, n).round().astype(int)
    return vol[idx]


class KneeDataset(Dataset):
    def __init__(self, uids, store, label_pack, image_size=IMAGE_SIZE,
                 n_slices=N_SLICES, augment=False):
        self.uids = uids
        self.store = store
        self.label_pack = label_pack
        self.image_size = image_size
        self.n_slices = n_slices
        self.augment = augment
        self.rng = np.random.RandomState(SEED)

    def __len__(self):
        return len(self.uids)

    def _augment_slice(self, img):
        # img: float32 (H, W) in [0, 1]. NO flips (medial != lateral).
        r = self.rng
        if r.rand() < 0.9:
            if r.rand() < 0.5:
                g = np.float32(0.85 + 0.30 * r.rand())
                img = np.clip(img * g, 0, 1).astype(np.float32)
            if r.rand() < 0.5:
                c = np.float32(0.8 + 0.45 * r.rand())
                img = np.clip((img - 0.5) * c + 0.5, 0, 1).astype(np.float32)
            if r.rand() < 0.5:
                gm = np.float32(0.7 + 0.7 * r.rand())
                img = np.clip(np.power(img, gm), 0, 1).astype(np.float32)
            if r.rand() < 0.3:
                img = np.clip(img + r.randn(*img.shape).astype(np.float32) * np.float32(0.02),
                              0, 1).astype(np.float32)
            if r.rand() < 0.3:
                h, w = img.shape
                ch = int(h * (0.2 + 0.3 * r.rand()))
                cw = int(w * (0.2 + 0.3 * r.rand()))
                y0 = int(r.rand() * max(1, h - ch))
                x0 = int(r.rand() * max(1, w - cw))
                img = img.copy()
                img[y0:y0+ch, x0:x0+cw] = 0
        return img.astype(np.float32)

    def _affine_vol(self, vol):
        r = self.rng
        ang = float(r.uniform(-15, 15)) * float(np.pi) / 180.0
        ca, sa = float(np.cos(ang)), float(np.sin(ang))
        zoom = float(0.90 + 0.25 * r.rand())
        tx = float(r.uniform(-0.08, 0.08))
        ty = float(r.uniform(-0.08, 0.08))
        theta = torch.tensor([[[zoom*ca, -zoom*sa, tx],
                               [zoom*sa,  zoom*ca, ty]]], dtype=torch.float32)
        t = torch.from_numpy(vol).unsqueeze(1).float()
        grid = F.affine_grid(theta.expand(t.size(0), -1, -1), t.size(),
                             align_corners=False).to(t.dtype)
        t = F.grid_sample(t, grid, mode='bilinear', padding_mode='zeros',
                          align_corners=False)
        return t.squeeze(1).numpy().astype(np.float32)

    def __getitem__(self, i):
        uid = self.uids[i]
        planes = {}
        mask = []
        have = set(self.store.planes_of(uid))
        for p in PLANES:
            if p in have:
                vol = self.store.get_plane(uid, p)  # (S, 336, 336) uint8
                vol = sample_slices(vol, self.n_slices).astype(np.float32) / 255.0
                if self.augment:
                    vol = np.stack([self._augment_slice(s) for s in vol])
                    if self.rng.rand() < 0.5:
                        vol = self._affine_vol(vol)
                t = torch.from_numpy(vol).unsqueeze(1)  # (S,1,H,W) float32
                t = F.interpolate(t, size=(self.image_size, self.image_size),
                                  mode='bilinear', align_corners=False)
                vol = t.squeeze(1).numpy().astype(np.float32)
                planes[p] = vol
                mask.append(1.0)
            else:
                planes[p] = np.zeros((self.n_slices, self.image_size,
                                      self.image_size), dtype=np.float32)
                mask.append(0.0)
        targets, loss_mask, sample_w, is_gold, ord_grades, ord_mask = self.label_pack[uid]
        planes3 = {p: np.repeat(v[:, None, :, :], 3, axis=1).astype(np.float32)
                   for p, v in planes.items()}
        return {
            'uid': uid,
            'planes': {p: torch.from_numpy(v) for p, v in planes3.items()},
            'plane_mask': torch.tensor(mask, dtype=torch.float32),
            'targets': torch.from_numpy(targets.astype(np.float32)),
            'loss_mask': torch.from_numpy(loss_mask.astype(np.float32)),
            'weight': torch.tensor(float(sample_w), dtype=torch.float32),
            'is_gold': torch.tensor(bool(is_gold)),
            'ord_grades': torch.from_numpy(np.asarray(ord_grades, dtype=np.float32)),
            'ord_mask': torch.from_numpy(np.asarray(ord_mask, dtype=np.float32)),
        }


# dataset self-test
_ds = KneeDataset(train_uids[:4], store, label_pack, augment=True)
_s = _ds[0]
print('sample keys:', sorted(_s.keys()))
for p in PLANES:
    print(f'  {p}: {tuple(_s["planes"][p].shape)}')
print(f'  plane_mask: {_s["plane_mask"].tolist()}')
print(f'  targets: {_s["targets"].shape} | loss_mask sum: {_s["loss_mask"].sum().item():.0f}/12')
print(f'  ord_grades: {_s["ord_grades"].tolist()} | ord_mask: {_s["ord_mask"].tolist()}')
assert _s['planes']['axial'].shape == (N_SLICES, 3, IMAGE_SIZE, IMAGE_SIZE)
assert _s['planes']['axial'].dtype == torch.float32
assert _s['ord_grades'].shape == (4,) and _s['ord_mask'].shape == (4,)
_miss = [u for u in train_uids if len(store.planes_of(u)) < 3]
if _miss:
    _d2 = KneeDataset(_miss[:1], store, label_pack)
    _s2 = _d2[0]
    pm = _s2['plane_mask']
    for p, m in zip(PLANES, pm.tolist()):
        if m == 0:
            assert (_s2['planes'][p] == 0).all(), f'{p} not zeroed'
    print(f'missing-plane study ok: mask {pm.tolist()} -> zeroed')
print('dataset self-test: ALL GREEN')


In [ ]:
# ================= MODEL: ConvNeXt-S + per-finding attention + CORAL + fastMRI aux =================
# Upgrade #2 (nartaa's trick): per-finding attention MIL — one shared projection
# (768->256, Tanh, Dropout 0.2) then a 256->12 head giving EACH finding its own
# attention weights over slices; einsum pooling. A cruciate tear may live on two
# sagittal slices while OA spreads across many coronal ones.
import timm

print('timm', timm.__version__)
assert BACKBONE_NAME in timm.list_models(), f'FATAL: {BACKBONE_NAME} not in this timm version'
print(f'creating {BACKBONE_NAME} (pretrained={PRETRAINED}, drop_path={DROP_PATH_RATE})...')


class PerFindingAttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=ATTN_DIM, n_findings=N_FINDINGS):
        super().__init__()
        self.fc1 = nn.Linear(feat_dim, attn_dim)
        self.drop = nn.Dropout(0.2)
        self.fc2 = nn.Linear(attn_dim, n_findings)

    def forward(self, h):  # h: (B, S, D)
        a = self.fc2(self.drop(torch.tanh(self.fc1(h))))  # (B, S, 12)
        a = F.softmax(a, dim=1)  # softmax over slices
        z = torch.einsum('bsf,bsd->bfd', a, h)  # (B, 12, D) per-finding pooled
        return z, a


class KneeConvNeXtS_MIL(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=PRETRAINED,
                 drop_path_rate=DROP_PATH_RATE):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_path_rate=drop_path_rate)
        print(f'backbone init OK: {backbone_name} pretrained={pretrained}')
        D = self.backbone.num_features
        print(f'backbone feature dim: {D}')
        assert D == 768, f'FATAL: per-finding attention spec needs 768-dim, got {D}'
        self.mil = PerFindingAttentionMIL(D, attn_dim, n_findings)
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])
        # Upgrade #1 (Justin's edge): CORAL ordinal heads for the 4 severity findings.
        # Each head emits 3 logits: P(y>0), P(y>1), P(y>2) on the finding's MIL feature.
        self.coral_heads = nn.ModuleList(
            [nn.Linear(D, N_CORAL_THRESHOLDS) for _ in CORAL_FINDING_IDX])
        # R7 trick: fastMRI auxiliary lesion-detection head on the shared encoder.
        self.fmri_head = nn.Sequential(
            nn.Linear(D, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))

    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        # reshape (not view): non-contiguous tensors crash view() on CUDA
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        z, _ = self.mil(h)
        return z  # (B, 12, D)

    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)  # (B, 12, D)
        logits = torch.cat([self.heads[f](fused[:, f, :])
                            for f in range(N_FINDINGS)], dim=1)  # (B, 12)
        coral_logits = torch.stack(
            [ch(fused[:, fi, :]) for fi, ch in zip(CORAL_FINDING_IDX, self.coral_heads)],
            dim=1)  # (B, 4, 3)
        return logits, coral_logits

    def encode_2d(self, x):  # x: (B, 3, H, W) fastMRI windows -> (B, D)
        return self.backbone(x)

    @property
    def enc(self):
        return self.backbone  # for frozen-BN helper


def freeze_bn_stats(model):
    # ConvNeXt uses LayerNorm (no running stats) — harmless no-op, kept for parity.
    n = 0
    for mod in model.enc.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.eval()
            n += 1
    return n


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


model = KneeConvNeXtS_MIL()
print(f'params: {count_params(model):.1f}M')
# CPU shape probe before GPU
model.eval()
with torch.no_grad():
    _pb = {p: torch.randn(1, 4, 3, IMAGE_SIZE, IMAGE_SIZE) for p in PLANES}
    _pm = torch.ones(1, 3)
    _o_bin, _o_coral = model(_pb, _pm)
    print(f'cpu shape probe: binary {tuple(_o_bin.shape)}, coral {tuple(_o_coral.shape)}')
    assert _o_bin.shape == (1, N_FINDINGS)
    assert _o_coral.shape == (1, len(CORAL_FINDING_IDX), N_CORAL_THRESHOLDS)
    _fx = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
    _fl = model.fmri_head(model.encode_2d(_fx))
    assert _fl.shape == (2, 1), f'fmri head shape {_fl.shape}'
    print(f'cpu shape probe: fmri aux {tuple(_fl.shape)} OK')
print('model self-test: ALL GREEN')
del model  # rebuilt in training cell / probe


In [ ]:
# ================= TRAINING UTILS: AdamW groups, warmup+cosine, EMA, AMP, SWA =================
import itertools

def make_loader(uids, augment, batch_size=BATCH_SIZE, shuffle=True, num_workers=4):
    ds = KneeDataset(uids, store, label_pack, augment=augment)

    def collate(batch):
        out = {'uid': [b['uid'] for b in batch]}
        out['planes'] = {p: torch.stack([b['planes'][p] for b in batch]) for p in PLANES}
        for k in ['plane_mask', 'targets', 'loss_mask', 'weight', 'is_gold',
                  'ord_grades', 'ord_mask']:
            out[k] = torch.stack([b[k] for b in batch])
        return out

    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True, collate_fn=collate)


def masked_asl(logits, targets, loss_mask, sample_weight):
    el = asl_loss(logits, targets)
    per_sample = (el * loss_mask).sum(1) / loss_mask.sum(1).clamp(min=1e-6)
    return (per_sample * sample_weight).mean()


def forward_loss(model, planes, plane_mask, targets, loss_mask, sample_weight,
                 ord_grades, ord_mask):
    # One canonical forward+loss path: backbone forward in AMP for speed,
    # ALL losses ALWAYS in fp32 (the ep16 NaN root cause was fp16 loss).
    # Adds LAMBDA_CORAL * CORAL ordinal loss on the 4 severity heads.
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        logits, coral_logits = model(planes, plane_mask)
    with torch.cuda.amp.autocast(enabled=False):
        loss = masked_asl(logits.float(), targets, loss_mask, sample_weight)
        coral_l = torch.tensor(0.0, device=logits.device)
        if ord_mask.sum() > 0:
            sev_targets = grade_to_coral_targets(ord_grades.float())  # (B,4,3)
            coral_l = coral_loss(coral_logits.float(), sev_targets, ord_mask.float())
            loss = loss + LAMBDA_CORAL * coral_l
    return loss, logits, coral_l


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, gm = [], [], []
    for b in loader:
        planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
        logits, coral_logits = model(planes, b['plane_mask'].to(DEVICE))
        probs = torch.sigmoid(logits).float()
        # CORAL inference boost: blend ordinal heads for 4 severity findings
        # CORAL_FINDING_IDX = [8, 6, 4, 5] -> synovitis, pfoa, medial_oa, lateral_oa
        if coral_logits is not None:
            coral_p = torch.sigmoid(coral_logits).float()  # (B, 4, 3)
            coral_mean = coral_p.mean(dim=2)  # (B, 4): avg P(grade>=k)
            for ci, fi in enumerate(CORAL_FINDING_IDX):
                probs[:, fi] = 0.5 * probs[:, fi] + 0.5 * coral_mean[:, ci]
        ys.append(b['targets'].numpy())
        ps.append(probs.cpu().numpy())
        gm.append(b['is_gold'].numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    gm = np.concatenate(gm).astype(bool)
    yb = (ys >= 0.5).astype(int)
    macro, per = macro_auc(yb, ps)
    macro_g, per_g = (None, {})
    if gm.sum() >= 4:
        macro_g, per_g = macro_auc(yb[gm], ps[gm])
    return macro, per, macro_g


def get_param_groups(model, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
                     weight_decay=WEIGHT_DECAY, stage_decay=LR_STAGE_DECAY):
    # Per-stage LR decay (TianK003 recipe): stages.3 gets lr_backbone, earlier
    # stages get x stage_decay each. MIL/heads/CORAL/fastMRI heads get lr_head.
    # No weight decay on biases, norms, 1-D params.
    import re as _re
    bucket = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith('backbone.stages.'):
            m = _re.match(r'backbone\.stages\.(\d+)\.', name)
            sidx = int(m.group(1)) if m else 0
            lr = lr_backbone * (stage_decay ** (3 - sidx))
            key = f'bb_stage{sidx}'
        elif name.startswith('backbone.'):
            lr = lr_backbone * (stage_decay ** 3)
            key = 'bb_stage0'
        else:
            lr = lr_head
            key = 'head'
        bucket.setdefault(key, (lr, []))[1].append((name, p))

    def split(params):
        decay = [p for n, p in params if p.dim() > 1]
        no_decay = [p for n, p in params if p.dim() <= 1]
        return decay, no_decay

    order = ['bb_stage0', 'bb_stage1', 'bb_stage2', 'bb_stage3', 'head']
    param_groups = []
    for key in order:
        if key not in bucket:
            continue
        lr, params = bucket[key]
        d, nd = split(params)
        if d:
            param_groups.append({'params': d, 'lr': lr, 'weight_decay': weight_decay})
        if nd:
            param_groups.append({'params': nd, 'lr': lr, 'weight_decay': 0.0})
    print(f'  param groups: {len(param_groups)} groups (per-stage decay {stage_decay})')
    return param_groups


def make_scheduler(opt, epochs, steps_per_epoch, warmup_frac=WARMUP_FRAC):
    total_steps = epochs * steps_per_epoch
    warmup_steps = max(1, int(warmup_frac * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(warmup_steps)
        prog = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda), total_steps, warmup_steps


class EMA:
    # Exponential moving average of model weights (decay 0.998). CPU shadow.
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().cpu(),
                                                     alpha=1.0 - self.decay)

    def state_dict(self):
        return {k: v.clone() for k, v in self.shadow.items()}


def average_state_dicts(sds):
    # manual SWA over EMA snapshots
    out = {}
    for k, v in sds[-1].items():
        if v.dtype.is_floating_point:
            out[k] = torch.stack([sd[k].float() for sd in sds]).mean(0).to(v.dtype)
        else:
            out[k] = v.clone()
    return out


def save_ckpt(path, model, opt, epoch, best, ema_sd=None, swa_sd=None, fold=None, sched=None):
    payload = {'epoch': epoch, 'best': float(best), 'fold': fold,
               'model': model.state_dict(),
               'opt': opt.state_dict() if opt is not None else None,
               'sched': sched.state_dict() if sched is not None else None}
    if ema_sd is not None:
        payload['ema'] = ema_sd
    if swa_sd is not None:
        payload['swa'] = swa_sd
    torch.save(payload, path)
    print(f'  saved {path}')


def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location='cpu')
    model.load_state_dict(ck['model'])
    if opt is not None and ck.get('opt') is not None:
        opt.load_state_dict(ck['opt'])
    print(f"  resumed {path} (epoch {ck['epoch']}, best {ck['best']}, fold {ck.get('fold')})")
    return ck['epoch'], ck['best'], ck.get('ema'), ck.get('swa'), ck.get('sched')


class FastMRIWindows(Dataset):
    # Random 3-slice windows from .npy volumes (D,H,W), binary lesion label.
    # Best-effort: only used if a fastMRI dataset is attached AND lesion labels exist.
    def __init__(self, items, image_size=IMAGE_SIZE, n_samples=4000, seed=SEED):
        self.items = items
        self.image_size = image_size
        self.n_samples = n_samples
        self.rng = np.random.RandomState(seed + 999)
        self._cache = {}

    def __len__(self):
        return self.n_samples

    def _load_vol(self, path):
        if path not in self._cache:
            v = np.load(path).astype(np.float32)
            while v.ndim > 3:
                v = v[0]
            assert v.ndim == 3, f'vol ndim {v.ndim}: {path}'
            lo, hi = np.percentile(v, [1, 99])
            v = np.clip((v - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
            if len(self._cache) > 8:
                self._cache.pop(next(iter(self._cache)))
            self._cache[path] = v
        return self._cache[path]

    def __getitem__(self, i):
        vpath, lpath = self.items[self.rng.randint(len(self.items))]
        vol = self._load_vol(vpath)
        D = vol.shape[0]
        s0 = self.rng.randint(0, max(1, D - 2))
        win = vol[s0:s0 + 3]
        if win.shape[0] < 3:
            win = np.repeat(win, 3, axis=0)[:3]
        t = torch.from_numpy(win).unsqueeze(0)  # (1,3,H,W)
        t = F.interpolate(t, size=(self.image_size, self.image_size),
                          mode='bilinear', align_corners=False)
        img = t.squeeze(0).numpy().astype(np.float32)  # (3,H,W)
        lv = np.load(lpath)
        lesion = 1.0 if float(np.asarray(lv).mean()) > 0.5 else 0.0
        return {'image': torch.from_numpy(img),
                'lesion': torch.tensor(lesion, dtype=torch.float32)}


def try_build_fmri_loader():
    # Best-effort fastMRI aux loader (R7 trick). NEVER raises: returns a
    # DataLoader or None. Training continues with a warning when unavailable.
    try:
        import glob as _glob
        roots = []
        for pat in ('/kaggle/input/*fastmri*', '/kaggle/input/*fastMRI*',
                    '/kaggle/input/*FastMRI*', '/kaggle/input/*fast_mri*'):
            roots += _glob.glob(pat)
        if not roots:
            print('  fastMRI: no *fastmri* dataset under /kaggle/input — aux DISABLED')
            return None
        vols = []
        for r in roots:
            vols += _glob.glob(os.path.join(r, '**', '*.npy'), recursive=True)
        vols = [v for v in vols if 'lesion' not in os.path.basename(v).lower()
                and 'mask' not in os.path.basename(v).lower()]
        if not vols:
            print(f'  fastMRI: dataset found but no .npy volumes — aux DISABLED')
            return None
        items = []
        for v in vols:
            base = os.path.splitext(v)[0]
            lab = None
            for cand in (base + '_lesion.npy', base + '_mask.npy'):
                if os.path.exists(cand):
                    lab = cand
                    break
            items.append((v, lab))
        n_lab = sum(1 for _, l in items if l is not None)
        if n_lab == 0:
            print(f'  fastMRI: {len(vols)} volumes but no lesion labels '
                  f'(*_lesion.npy siblings) — aux DISABLED')
            return None
        print(f'  fastMRI: {len(vols)} volumes, {n_lab} with lesion labels — aux ENABLED')
        ds = FastMRIWindows(items, image_size=IMAGE_SIZE)
        return DataLoader(ds, batch_size=32, shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=True)
    except Exception as e:
        print(f'  fastMRI: loader build failed ({e!r}) — aux DISABLED')
        return None

print('training utils: loaded')



In [ ]:
# ================= PROBE MODE: full pipeline on tiny subset =================
# Gate: this cell must print ALL GREEN before any full run.
if PROBE:
    print('=== PROBE: forward + backward + AMP + EMA + CORAL + fmri-head + ckpt ===')
    pm = KneeConvNeXtS_MIL().to(DEVICE)
    popt = torch.optim.AdamW(get_param_groups(pm), lr=LR_BACKBONE)
    pscaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    pema = EMA(pm)
    ptr = make_loader(train_uids, augment=True, batch_size=2, shuffle=True, num_workers=0)
    pva = make_loader(val_uids, augment=False, batch_size=2, shuffle=False, num_workers=0)
    b = next(iter(ptr))
    assert b['planes']['axial'].dtype == torch.float32, 'FATAL: float64 leak in dataset'
    assert b['ord_grades'].shape[1] == 4 and b['ord_mask'].shape[1] == 4, 'FATAL: ord shape'
    print(f"  batch planes: {tuple(b['planes']['axial'].shape)} | "
          f"ord {tuple(b['ord_grades'].shape)} mask_sum {b['ord_mask'].sum().item():.0f}")
    print('  DataParallel: SKIPPED (known unsafe on T4, single-GPU only)')

    planes = {p: v.to(DEVICE) for p, v in b['planes'].items()}
    pm.train()
    freeze_bn_stats(pm)
    popt.zero_grad()
    loss, logits, coral_l = forward_loss(
        pm, planes, b['plane_mask'].to(DEVICE),
        b['targets'].to(DEVICE), b['loss_mask'].to(DEVICE), b['weight'].to(DEVICE),
        b['ord_grades'].to(DEVICE), b['ord_mask'].to(DEVICE))
    assert logits.shape == (2, N_FINDINGS), f'bad logits shape {logits.shape}'
    assert torch.isfinite(loss) and torch.isfinite(coral_l), 'FATAL: NaN loss'
    print(f'  forward+AMP: {tuple(logits.shape)} | loss {loss.item():.4f} '
          f'(coral {coral_l.item():.4f})')
    pscaler.scale(loss).backward()
    # fastMRI aux head: synthetic shape/grad test (no real data needed in probe)
    _fx = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        _ff = pm.encode_2d(_fx)
        _fl = pm.fmri_head(_ff)
    assert _fl.shape == (2, 1), f'FATAL: fmri head shape {_fl.shape}'
    _fml = F.binary_cross_entropy_with_logits(
        _fl.float(), torch.ones(2, 1, device=DEVICE),
        pos_weight=torch.tensor([FMRI_POS_WEIGHT], device=DEVICE))
    assert torch.isfinite(_fml), 'FATAL: NaN fmri aux loss'
    pscaler.scale(_fml).backward()
    print(f'  fmri aux head OK (synthetic): {_fl.shape} | aux loss {_fml.item():.4f}')
    del _fx, _ff, _fl, _fml
    pscaler.unscale_(popt)
    gn = torch.nn.utils.clip_grad_norm_(pm.parameters(), GRAD_CLIP)
    pscaler.step(popt); pscaler.update(); pema.update(pm)
    print(f'  backward+step+EMA OK (grad norm {gn:.2f})')
    if torch.cuda.is_available():
        print(f"  gpu mem peak: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB (T4 = 14.56)")

    macro, per, macro_g = evaluate(pm, pva)
    print(f'  val macro AUC: {macro:.4f}' if macro else '  val macro AUC: n/a')
    save_ckpt(f'{CKPT_DIR}/probe_roundtrip.pt', pm, popt, 0, macro or -1,
              ema_sd=pema.state_dict(), fold=FOLD)
    _m2 = KneeConvNeXtS_MIL().to(DEVICE)
    load_ckpt(f'{CKPT_DIR}/probe_roundtrip.pt', _m2, None)
    print('  ckpt roundtrip OK')
    del pm, _m2
    print('PROBE: ALL GREEN')
else:
    print('(probe mode off — full training next)')


In [ ]:
# ================= TRAINING: 30 epochs, per-stage AdamW, warmup+cosine, AMP, EMA =================
def train_fold(uids_tr, uids_va, epochs=EPOCHS, patience=PATIENCE,
               prefix=None):
    if prefix is None:
        prefix = f'convnext_s_fold{FOLD}'
    model = KneeConvNeXtS_MIL().to(DEVICE)
    # Single-GPU only (CoAtNet proved DP-unsafe twice; no DataParallel anywhere).
    opt = torch.optim.AdamW(get_param_groups(model), lr=LR_BACKBONE)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    ema = EMA(model, decay=EMA_DECAY)
    ema_snaps = []  # ring buffer of last SWA_LAST EMA snapshots (CPU)

    # CHECKPOINT RESUME: if _last.pt exists, resume from it (9h Kaggle limit)
    start_ep, best, bad = 0, -1.0, 0
    _resume_path = f'{CKPT_DIR}/{prefix}_last.pt'
    _sched_state = None
    if os.path.exists(_resume_path):
        try:
            _ep, _best, _ema_sd, _swa_sd, _sched_state = load_ckpt(_resume_path, model, opt)
            if _ema_sd is not None:
                ema.load_state_dict(_ema_sd)
            start_ep, best = _ep + 1, _best
            print(f'  RESUME: continuing from epoch {start_ep} (best {best:.4f})')
        except Exception as e:
            print(f'  resume failed ({e}), starting fresh')

    # fastMRI auxiliary (R7 trick): None-safe; training continues if unavailable.
    fmri_loader = try_build_fmri_loader()
    fmri_iter = itertools.cycle(fmri_loader) if fmri_loader is not None else None
    fmri_on = fmri_iter is not None
    print(f'  fastMRI aux: {"ENABLED" if fmri_on else "DISABLED"}')

    tr_loader = make_loader(uids_tr, augment=True)
    va_loader = make_loader(uids_va, augment=False, shuffle=False)
    steps_per_epoch = math.ceil(len(uids_tr) / BATCH_SIZE)
    opt_steps_per_epoch = math.ceil(steps_per_epoch / ACCUM_STEPS)
    sched, total_steps, warmup_steps = make_scheduler(opt, epochs, opt_steps_per_epoch)
    if _sched_state is not None:
        try:
            sched.load_state_dict(_sched_state)
            print(f'  scheduler restored')
        except Exception as e:
            print(f'  scheduler restore failed ({e})')
    print(f'fold {FOLD} [{prefix}] | train: {len(uids_tr)} / val: {len(uids_va)} | '
          f'{steps_per_epoch} batches/ep x {epochs} eps (patience {patience}) | '
          f'{opt_steps_per_epoch} opt-steps/ep | warmup {warmup_steps}')
    print(f'AMP: {USE_AMP} | EMA: {EMA_DECAY} | CORAL aux: {LAMBDA_CORAL} | '
          f'fastMRI aux: {FASTMRI_AUX_W}')

    for ep in range(start_ep, epochs):
        model.train()
        if FREEZE_BN:
            freeze_bn_stats(model)
        t_ep, tot, tot_coral, n = time.time(), 0.0, 0.0, 0
        opt.zero_grad()
        for si, b in enumerate(tr_loader):
            planes = {p: v.to(DEVICE, non_blocking=True) for p, v in b['planes'].items()}
            loss, _, coral_l = forward_loss(
                model, planes, b['plane_mask'].to(DEVICE),
                b['targets'].to(DEVICE), b['loss_mask'].to(DEVICE),
                b['weight'].to(DEVICE),
                b['ord_grades'].to(DEVICE), b['ord_mask'].to(DEVICE))
            if not torch.isfinite(loss):
                raise RuntimeError(f'KILL GATE: NaN/Inf loss at ep{ep} step{si}')
            scaler.scale(loss / ACCUM_STEPS).backward()
            # R7: fastMRI aux runs AFTER the main backward, every micro-batch.
            if fmri_on:
                try:
                    fb = next(fmri_iter)
                    with torch.cuda.amp.autocast(enabled=USE_AMP):
                        _ff = model.encode_2d(fb['image'].to(DEVICE, non_blocking=True))
                        _fl = model.fmri_head(_ff)
                    with torch.cuda.amp.autocast(enabled=False):
                        _fml = F.binary_cross_entropy_with_logits(
                            _fl.float(), fb['lesion'].to(DEVICE).float().unsqueeze(1),
                            pos_weight=torch.tensor([FMRI_POS_WEIGHT], device=DEVICE))
                    scaler.scale(FASTMRI_AUX_W * _fml / ACCUM_STEPS).backward()
                    del _ff, _fl, _fml
                except Exception as e:
                    print(f'  fastMRI aux disabled mid-run: {e!r}')
                    fmri_on = False
            if (si + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad()
                ema.update(model); sched.step()
            tot += loss.item(); tot_coral += coral_l.item(); n += 1
        # leftover accum
        if n % ACCUM_STEPS:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            ema.update(model); sched.step()

        # EMA snapshot at epoch end (CPU ring buffer)
        ema_snaps.append(ema.state_dict())
        ema_snaps = ema_snaps[-SWA_LAST:]

        macro, per, macro_g = evaluate(model, va_loader)
        _ema_model = KneeConvNeXtS_MIL().to(DEVICE)
        _ema_model.load_state_dict(ema.state_dict(), strict=False)
        macro_ema, _, _ = evaluate(_ema_model, va_loader)
        del _ema_model
        dt = time.time() - t_ep
        msg = f'  ep{ep+1}/{epochs} | loss {tot/n:.4f} (coral {tot_coral/n:.4f})'
        msg += f' | val {macro:.4f}' if macro else ' | val n/a'
        msg += f' | ema {macro_ema:.4f}' if macro_ema else ''
        msg += f' | gold {macro_g:.4f}' if macro_g else ''
        print(msg + f' | {dt:.0f}s', flush=True)

        improved = macro is not None and macro > best
        if improved:
            best, bad = macro, 0
            save_ckpt(f'{CKPT_DIR}/{prefix}_best.pt', model, opt, ep, best,
                      ema_sd=ema.state_dict(), fold=FOLD, sched=sched)
        else:
            bad += 1
        save_ckpt(f'{CKPT_DIR}/{prefix}_last.pt', model, opt, ep, best,
                  ema_sd=ema.state_dict(), fold=FOLD, sched=sched)
        if bad >= patience:
            print(f'  early stopping at ep{ep+1}: no val improvement for {patience} epochs')
            break
    return f'{CKPT_DIR}/{prefix}_best.pt', best, ema_snaps


if not PROBE:
    best_ckpt, best_val, ema_snaps = train_fold(train_uids, val_uids)
    print(f'TRAINING done: fold {FOLD} best val macro AUC {best_val:.4f} -> {best_ckpt}')
else:
    print(f'(probe mode: {PROBE_EPOCHS} epochs on {len(train_uids)} studies, '
          f'separate probe ckpts)')
    best_ckpt, best_val, ema_snaps = train_fold(
        train_uids, val_uids, epochs=PROBE_EPOCHS, patience=PROBE_EPOCHS + 1,
        prefix='convnext_s_probe')
    print(f'PROBE training done: best val {best_val:.4f}' if best_val and best_val > 0
          else 'PROBE training done')



In [ ]:
# ================= SWA: average last SWA_LAST EMA snapshots =================
if not PROBE:
    assert len(ema_snaps) == SWA_LAST, \
        f'FATAL: need {SWA_LAST} EMA snapshots, have {len(ema_snaps)}'
    print(f'averaging {len(ema_snaps)} EMA snapshots...')
    swa_sd = average_state_dicts(ema_snaps)
    _k = next(k for k, v in swa_sd.items() if v.dtype.is_floating_point)
    _diff = (swa_sd[_k] - ema_snaps[-1][_k]).abs().mean().item()
    print(f'  mean abs diff vs last EMA snapshot: {_diff:.2e} (expect small, nonzero)')

    swa_model = KneeConvNeXtS_MIL().to(DEVICE)
    swa_model.load_state_dict(swa_sd)
    va_loader = make_loader(val_uids, augment=False, shuffle=False)
    macro_swa, per_swa, macro_g_swa = evaluate(swa_model, va_loader)
    print(f'  SWA val macro: {macro_swa:.4f}' if macro_swa else '  SWA val macro: n/a',
          f'| gold-macro {macro_g_swa:.4f}' if macro_g_swa else '')

    torch.save({'swa': swa_sd, 'val_macro': float(macro_swa or -1), 'fold': FOLD},
               f'{CKPT_DIR}/convnext_s_fold{FOLD}_swa.pt')
    print(f'\nFINAL ARTIFACTS in {CKPT_DIR}/:')
    print(f'  convnext_s_fold{FOLD}_best.pt / _last.pt / _swa.pt')
    print('Upload the _best.pt (or _swa.pt) as a Kaggle dataset before closing this session.')
else:
    print('(probe mode: SWA skipped)')
